# Clean Knowledge Distillation Pipeline Demo

This notebook demonstrates a streamlined knowledge distillation pipeline that uses a pre-trained ResNet50 teacher model to train a lightweight student model.

## Features:
- Uses existing trained teacher model (`best_teacher_model.pth`)
- Balanced sampling for handling class imbalance
- Advanced data augmentation
- Clean, minimal dependencies
- Comprehensive loss tracking (soft + hard targets)
- Real-time training visualization


In [ ]:
# Import required libraries
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Add current directory to path
sys.path.append(str(Path.cwd()))

# Import our clean distillation pipeline
from clean_distillation_pipeline import (
    CleanDistillationPipeline,
    load_dataset_from_csv
)

print("✅ Libraries imported successfully!")


## 1. Load Dataset

Load your dataset from CSV file. The pipeline will automatically detect class columns and handle the data loading.


In [ ]:
# Configuration
teacher_model_path = "best_teacher_model.pth"  # Path to your trained teacher model
csv_path = "./data/annotations.csv"  # Path to your dataset CSV
data_root = "./data"  # Root directory for images

# Load dataset
print("Loading dataset...")
df, class_names, class_id_col = load_dataset_from_csv(csv_path, data_root)
num_classes = len(class_names)

print(f"\nDataset Summary:")
print(f"Total samples: {len(df)}")
print(f"Number of classes: {num_classes}")
print(f"Class names: {class_names}")
print(f"Class ID column: {class_id_col}")


## 2. Initialize Distillation Pipeline

Create the distillation pipeline with your trained teacher model.


In [ ]:
# Initialize distillation pipeline
print("Initializing distillation pipeline...")
pipeline = CleanDistillationPipeline(
    teacher_model_path=teacher_model_path,
    num_classes=num_classes,
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

print(f"\nPipeline initialized successfully!")
print(f"Device: {pipeline.device}")
print(f"Number of classes: {num_classes}")


## 3. Create Data Loaders

Create training, validation, and test data loaders with balanced sampling and augmentation.


In [ ]:
# Create data loaders
print("Creating data loaders...")
train_loader, val_loader, test_loader, train_df, val_df, test_df = pipeline.create_data_loaders(
    df, 
    data_root=data_root, 
    batch_size=32, 
    test_size=0.2, 
    val_size=0.1, 
    class_id_col=class_id_col
)

print(f"\nData Split Summary:")
print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Test samples: {len(test_df)}")
print(f"Total samples: {len(train_df) + len(val_df) + len(test_df)}")

# Show class distribution in each split
print(f"\nClass Distribution:")
for split_name, split_df in [("Training", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(f"\n{split_name} set:")
    split_counts = split_df[class_id_col].value_counts().sort_index()
    for class_id, count in split_counts.items():
        class_name = class_names[class_id] if class_id < len(class_names) else f"class_{class_id}"
        percentage = (count / len(split_df)) * 100
        print(f"  {class_name}: {count} samples ({percentage:.1f}%)")


## 4. Train Student Model

Train the lightweight student model using knowledge distillation from the teacher model.


In [ ]:
# Train student model
print("Starting distillation training...")
print("This will train the student model using knowledge from the teacher model.")
print("The loss combines both soft targets (teacher predictions) and hard targets (ground truth).")

history, best_val_f1 = pipeline.train_student(
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=30,  # Adjust as needed
    learning_rate=1e-3,
    weight_decay=1e-4
)

print(f"\nTraining completed!")
print(f"Best validation F1 score: {best_val_f1:.4f}")


## 5. Visualize Training Progress

Plot the training history to see how the student model learned from the teacher.


In [ ]:
# Plot training history
pipeline.plot_training_history(history)

# Print final training metrics
print("\nFinal Training Metrics:")
print(f"Final training accuracy: {history['train_acc'][-1]:.4f}")
print(f"Final validation accuracy: {history['val_acc'][-1]:.4f}")
print(f"Final validation F1: {history['val_f1'][-1]:.4f}")
print(f"Best validation F1: {best_val_f1:.4f}")

# Show loss components
print(f"\nLoss Components (Final Epoch):")
print(f"Soft loss (teacher knowledge): {history['soft_loss'][-1]:.4f}")
print(f"Hard loss (ground truth): {history['hard_loss'][-1]:.4f}")
print(f"Total loss: {history['train_loss'][-1]:.4f}")


## 6. Evaluate Models

Compare the performance of the teacher and student models on the test set.


In [ ]:
# Evaluate both models
print("Evaluating teacher and student models...")
results = pipeline.evaluate_models(test_loader)

# Display results
print("\n=== Model Comparison Results ===")
print(f"Teacher Model:")
print(f"  Accuracy: {results['teacher']['accuracy']:.4f}")
print(f"  F1 Score: {results['teacher']['f1_score']:.4f}")
print(f"  Parameters: {results['teacher']['parameters']:,}")

print(f"\nStudent Model:")
print(f"  Accuracy: {results['student']['accuracy']:.4f}")
print(f"  F1 Score: {results['student']['f1_score']:.4f}")
print(f"  Parameters: {results['student']['parameters']:,}")

print(f"\nCompression Metrics:")
print(f"  Compression Ratio: {results['compression_ratio']:.1f}x")
print(f"  Performance Retention: {results['performance_retention']:.2%}")

# Calculate efficiency metrics
teacher_efficiency = results['teacher']['f1_score'] / results['teacher']['parameters'] * 1e6
student_efficiency = results['student']['f1_score'] / results['student']['parameters'] * 1e6
efficiency_gain = student_efficiency / teacher_efficiency

print(f"\nEfficiency Metrics:")
print(f"  Teacher Efficiency: {teacher_efficiency:.6f} F1 per million parameters")
print(f"  Student Efficiency: {student_efficiency:.6f} F1 per million parameters")
print(f"  Efficiency Gain: {efficiency_gain:.1f}x")


## 7. Create Comparison Visualization

Create a comprehensive visualization comparing the teacher and student models.


In [ ]:
# Create comparison visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Teacher vs Student Model Comparison', fontsize=16, fontweight='bold')

# 1. Performance metrics comparison
metrics = ['Accuracy', 'F1 Score']
teacher_values = [results['teacher']['accuracy'], results['teacher']['f1_score']]
student_values = [results['student']['accuracy'], results['student']['f1_score']]

x = np.arange(len(metrics))
width = 0.35

axes[0, 0].bar(x - width/2, teacher_values, width, label='Teacher (ResNet50)', alpha=0.8, color='skyblue')
axes[0, 0].bar(x + width/2, student_values, width, label='Student (Lightweight)', alpha=0.8, color='lightcoral')
axes[0, 0].set_title('Performance Comparison')
axes[0, 0].set_ylabel('Score')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(metrics)
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Parameter count comparison
param_counts = [results['teacher']['parameters'], results['student']['parameters']]
model_names = ['Teacher', 'Student']
colors = ['skyblue', 'lightcoral']

axes[0, 1].bar(model_names, param_counts, color=colors, alpha=0.8)
axes[0, 1].set_title('Parameter Count Comparison')
axes[0, 1].set_ylabel('Parameters')
axes[0, 1].set_yscale('log')
for i, v in enumerate(param_counts):
    axes[0, 1].text(i, v * 1.1, f'{v:,}', ha='center', va='bottom')

# 3. Compression metrics
compression_metrics = ['Compression\nRatio', 'Performance\nRetention']
compression_values = [results['compression_ratio'], results['performance_retention'] * 100]
compression_colors = ['gold', 'lightgreen']

axes[1, 0].bar(compression_metrics, compression_values, color=compression_colors, alpha=0.8)
axes[1, 0].set_title('Compression & Performance Metrics')
axes[1, 0].set_ylabel('Value')
for i, v in enumerate(compression_values):
    unit = 'x' if i == 0 else '%'
    axes[1, 0].text(i, v + 0.5, f'{v:.1f}{unit}', ha='center', va='bottom')

# 4. Efficiency comparison
efficiency_values = [teacher_efficiency, student_efficiency]
axes[1, 1].bar(model_names, efficiency_values, color=colors, alpha=0.8)
axes[1, 1].set_title('Efficiency (F1 Score per Million Parameters)')
axes[1, 1].set_ylabel('F1 Score / Million Params')
for i, v in enumerate(efficiency_values):
    axes[1, 1].text(i, v + 0.001, f'{v:.4f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

print("\n=== Summary ===")
print(f"✅ Successfully created a {results['compression_ratio']:.1f}x smaller model")
print(f"✅ Retained {results['performance_retention']:.1%} of teacher performance")
print(f"✅ Achieved {efficiency_gain:.1f}x efficiency improvement")
print(f"✅ Student model is ready for production deployment!")


## 8. Save Student Model

Save the trained student model for future use.


In [ ]:
# Save the trained student model
student_model_path = "best_student_model.pth"
pipeline.save_student_model(student_model_path)

print(f"\nStudent model saved to: {student_model_path}")
print("\nYou can now use this lightweight model for production deployment!")

# Show file size comparison
import os
teacher_size = os.path.getsize(teacher_model_path) / (1024 * 1024)  # MB
student_size = os.path.getsize(student_model_path) / (1024 * 1024)  # MB

print(f"\nFile Size Comparison:")
print(f"Teacher model: {teacher_size:.2f} MB")
print(f"Student model: {student_size:.2f} MB")
print(f"Size reduction: {(teacher_size - student_size) / teacher_size * 100:.1f}%")


## 9. Key Insights

### 🎯 **Distillation Success Metrics**
- **Performance Retention**: The student model achieves a significant portion of teacher performance
- **Size Reduction**: Dramatic reduction in model size and parameters
- **Efficiency**: Student model is much more efficient per parameter
- **Loss Components**: The training shows how both soft (teacher) and hard (ground truth) targets contribute to learning

### 🚀 **Production Benefits**
1. **Mobile Deployment**: Lightweight model suitable for mobile devices
2. **Edge Computing**: Can run on resource-constrained environments
3. **Cost Reduction**: Lower inference costs due to smaller model size
4. **Real-time Performance**: Faster inference due to reduced complexity

### 🔧 **Technical Features**
1. **Balanced Sampling**: Handles class imbalance during training
2. **Advanced Augmentation**: Random masking, rotation, color jitter
3. **Comprehensive Loss**: Combines teacher knowledge with ground truth
4. **Real-time Monitoring**: Track both soft and hard loss components

### 📈 **Next Steps**
1. **Quantization**: Apply post-training quantization for further compression
2. **Pruning**: Remove unnecessary connections for additional size reduction
3. **Hardware Optimization**: Optimize for specific deployment targets
4. **Continuous Learning**: Implement online learning for model updates

This clean pipeline demonstrates the power of knowledge distillation for creating production-ready models that balance performance and efficiency!
